# Rasters satellites avant / après inondation + tables attributaires (Casablanca)

Ce notebook (à exécuter sur **Google Colab**) fait deux choses :

**Partie A – Récupération avec Google Earth Engine**
- composites **avant** et **après** chaque événement (Sentinel-1 radar + Sentinel-2 optique pour janvier 2021, Landsat 5 pour novembre 2010) ;
- un masque d'inondation par détection de changement (0 = non inondé, 1 = inondé) ;
- MNT et pente (SRTM) ;
- export de tous les rasters en GeoTIFF (EPSG:26191) vers Google Drive.

**Partie B – Tables attributaires**
Un GeoTIFF n'a pas de table attributaire en natif. Pour chaque raster, on construit donc :
- une **table des bandes** (min, max, moyenne, écart-type, % de pixels valides) ;
- pour les masques, une **table des valeurs** à la ArcGIS (Value, Count, surface en km²) ;
- un **échantillon de pixels** (x, y + valeur de chaque bande), exportable en CSV/Excel.

Les fenêtres de dates sont basées sur la presse : pluies diluviennes la nuit du **29 au 30 novembre 2010**, et fortes pluies le **5 janvier 2021** (averses jusqu'au 8). Vérifiez avec la cellule « dates disponibles » qu'il existe des images satellites juste après chaque pluie.

## Partie A – Google Earth Engine

In [ ]:
# Colab fournit déjà pandas, numpy, scikit-learn... : on installe seulement rasterio.
# Si Colab signale une erreur numpy, faites Exécution > Redémarrer la session puis relancez sans cette cellule.
!pip install -q rasterio

In [ ]:
PROJET_GEE = "VOTRE_PROJET_GEE"          # identifiant de votre projet Google Cloud / Earth Engine
ROI_COORDS = [-7.85, 33.45, -7.35, 33.70]  # [ouest, sud, est, nord] en degrés : emprise approximative, à ajuster
DRIVE_FOLDER = "casa_flood_rasters"        # dossier créé dans Google Drive
CRS_EXPORT = "EPSG:26191"                  # Merchich / Nord Maroc
SCALE_S1S2 = 20                            # mètres (10 = plus précis mais fichiers 4 fois plus lourds)
SCALE_L5 = 30
ORBITE_S1 = None                           # "ASCENDING", "DESCENDING" ou None (toutes)

# Fenêtres de dates (début inclus, fin exclue), d'après la presse :
#  - 2010 : pluies diluviennes la nuit du 29 au 30 novembre 2010 (178 mm à Casa-Anfa)
#  - 2021 : fortes pluies le 5 janvier 2021 (16 h-22 h), averses jusqu'au 8 janvier
# L'« après » commence juste après la pluie : l'eau se retire vite, il faut une image proche de l'événement.
# Si une fenêtre ne contient aucune image, la cellule « dates disponibles » vous le dira : élargissez-la.
EVENEMENTS = {
    "2021_janvier":  dict(capteur="S1", avant=("2020-12-15", "2021-01-05"), apres=("2021-01-06", "2021-01-13")),
    "2010_novembre": dict(capteur="L5", avant=("2010-10-01", "2010-11-29"), apres=("2010-11-30", "2010-12-31")),
}

In [ ]:
import ee
try:
    ee.Initialize(project=PROJET_GEE)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJET_GEE)
ROI = ee.Geometry.Rectangle(ROI_COORDS)
print("Earth Engine initialisé")

### Fonctions : composites et masques d'inondation

In [ ]:
def verifier(col, libelle):
    n = col.size().getInfo()
    print(f"{libelle} : {n} image(s)")
    if n == 0:
        raise ValueError(f"Aucune image pour {libelle} : élargissez la fenêtre de dates.")

# --- Sentinel-2 (optique) ---
def s2_composite(d0, d1):
    def masque(img):
        scl = img.select("SCL")
        ok = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
        return img.updateMask(ok)
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(ROI).filterDate(d0, d1)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40)).map(masque))
    verifier(col, f"Sentinel-2 {d0} -> {d1}")
    img = (col.median().select(["B2", "B3", "B4", "B8", "B11", "B12"],
                               ["bleu", "vert", "rouge", "nir", "swir1", "swir2"]).divide(10000))
    idx = [img.normalizedDifference(["nir", "rouge"]).rename("ndvi"),
           img.normalizedDifference(["swir1", "nir"]).rename("ndbi"),
           img.normalizedDifference(["vert", "swir1"]).rename("mndwi")]
    return img.addBands(idx).toFloat().clip(ROI)

# --- Sentinel-1 (radar, dB) ---
def s1_composite(d0, d1):
    col = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(ROI).filterDate(d0, d1)
           .filter(ee.Filter.eq("instrumentMode", "IW"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))
    if ORBITE_S1:
        col = col.filter(ee.Filter.eq("orbitProperties_pass", ORBITE_S1))
    verifier(col, f"Sentinel-1 {d0} -> {d1}")
    return col.select(["VV", "VH"]).median().rename(["vv_db", "vh_db"]).toFloat().clip(ROI)

# --- Landsat 5 (optique, 30 m) ---
def l5_composite(d0, d1):
    def masque(img):
        qa = img.select("QA_PIXEL")
        return img.updateMask(qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0)))
    col = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2").filterBounds(ROI).filterDate(d0, d1).map(masque)
    verifier(col, f"Landsat 5 {d0} -> {d1}")
    img = (col.median().select(["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"],
                               ["bleu", "vert", "rouge", "nir", "swir1", "swir2"])
           .multiply(0.0000275).add(-0.2))
    idx = [img.normalizedDifference(["nir", "rouge"]).rename("ndvi"),
           img.normalizedDifference(["swir1", "nir"]).rename("ndbi"),
           img.normalizedDifference(["vert", "swir1"]).rename("mndwi")]
    return img.addBands(idx).toFloat().clip(ROI)

# --- Masques d'inondation (0 = non inondé, 1 = inondé, 255 = sans donnée) ---
def masque_s1(avant, apres):
    diff = apres.select("vv_db").subtract(avant.select("vv_db"))
    eau_perm = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").gt(50).unmask(0)
    pente = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003"))
    inonde = diff.lt(-3).And(apres.select("vv_db").lt(-15)).And(eau_perm.Not()).And(pente.lt(5))
    return inonde.rename("inonde").toByte().clip(ROI)

def masque_l5(avant, apres):
    inonde = apres.select("mndwi").gt(0).And(avant.select("mndwi").lte(0))
    return inonde.rename("inonde").toByte().clip(ROI)

In [ ]:
import time
def dates_disponibles(coll_id, d0, d1, filtre=None):
    col = ee.ImageCollection(coll_id).filterBounds(ROI).filterDate(d0, d1)
    if filtre is not None:
        col = col.filter(filtre)
    t = col.aggregate_array("system:time_start").getInfo()
    return sorted({time.strftime("%Y-%m-%d", time.gmtime(x / 1000)) for x in t})

for evt, p in EVENEMENTS.items():
    for phase in ("avant", "apres"):
        d0, d1 = p[phase]
        if p["capteur"] == "S1":
            print(evt, phase, "Sentinel-1 :", dates_disponibles("COPERNICUS/S1_GRD", d0, d1, ee.Filter.eq("instrumentMode", "IW")))
        else:
            print(evt, phase, "Landsat 5  :", dates_disponibles("LANDSAT/LT05/C02/T1_L2", d0, d1))

### Dates d'images disponibles
La cellule ci-dessus liste les dates d'acquisition réellement disponibles pour chaque fenêtre. Pour l'« après », il faut au moins une date juste après la pluie ; sinon élargissez la fenêtre de quelques jours.

### Construction des images
Seuils par défaut du masque Sentinel-1 : baisse de rétrodiffusion VV de plus de 3 dB, VV après < -15 dB, hors eau permanente (JRC) et pente < 5°. Pour Landsat 5 : MNDWI > 0 après et ≤ 0 avant. Ce sont des règles simples à valider et à affiner avec votre propre méthode.

In [ ]:
IMAGES = {}   # nom -> (image, résolution en m)

for evt, p in EVENEMENTS.items():
    if p["capteur"] == "S1":
        s1_av, s1_ap = s1_composite(*p["avant"]), s1_composite(*p["apres"])
        s2_av, s2_ap = s2_composite(*p["avant"]), s2_composite(*p["apres"])
        IMAGES[f"{evt}_s1_avant"] = (s1_av, SCALE_S1S2)
        IMAGES[f"{evt}_s1_apres"] = (s1_ap, SCALE_S1S2)
        IMAGES[f"{evt}_s1_diff"] = (s1_ap.subtract(s1_av).rename(["diff_vv", "diff_vh"]), SCALE_S1S2)
        IMAGES[f"{evt}_s2_avant"] = (s2_av, SCALE_S1S2)
        IMAGES[f"{evt}_s2_apres"] = (s2_ap, SCALE_S1S2)
        IMAGES[f"masque_{evt}"] = (masque_s1(s1_av, s1_ap), SCALE_S1S2)
    elif p["capteur"] == "L5":
        l5_av, l5_ap = l5_composite(*p["avant"]), l5_composite(*p["apres"])
        IMAGES[f"{evt}_l5_avant"] = (l5_av, SCALE_L5)
        IMAGES[f"{evt}_l5_apres"] = (l5_ap, SCALE_L5)
        IMAGES[f"masque_{evt}"] = (masque_l5(l5_av, l5_ap), SCALE_L5)

dem = ee.Image("USGS/SRTMGL1_003").clip(ROI).rename("dem")
IMAGES["dem_pente"] = (dem.addBands(ee.Terrain.slope(dem).rename("pente")).toFloat(), 30)
IMAGES["eau_permanente"] = (ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").gt(50).unmask(0).rename("eau").toFloat().clip(ROI), SCALE_S1S2)

print(len(IMAGES), "rasters prêts :", list(IMAGES))

### Export vers Google Drive

In [ ]:
TACHES, BANDES = {}, {}
for nom, (img, sc) in IMAGES.items():
    nodata = 255 if nom.startswith("masque_") else -9999
    BANDES[nom] = img.bandNames().getInfo()
    t = ee.batch.Export.image.toDrive(image=img.unmask(nodata), description=nom, folder=DRIVE_FOLDER,
                                      fileNamePrefix=nom, region=ROI, scale=sc, crs=CRS_EXPORT,
                                      maxPixels=1e10, fileFormat="GeoTIFF")
    t.start()
    TACHES[nom] = t
print(len(TACHES), "exports lancés vers Drive/", DRIVE_FOLDER)

In [ ]:
import time
from collections import Counter
while True:
    etats = {n: t.status() for n, t in TACHES.items()}
    cnt = Counter(s["state"] for s in etats.values())
    print(time.strftime("%H:%M:%S"), dict(cnt))
    if all(s["state"] in ("COMPLETED", "FAILED", "CANCELLED") for s in etats.values()):
        break
    time.sleep(30)
for n, s in etats.items():
    if s["state"] != "COMPLETED":
        print("⚠", n, s["state"], s.get("error_message", ""))

## Partie B – Tables attributaires
Si vous avez déjà téléchargé vos rasters (autre session ou téléchargement manuel), commencez directement ici et indiquez simplement le dossier dans `DOSSIER`.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DOSSIER = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
except ImportError:
    DOSSIER = DRIVE_FOLDER        # exécution locale : dossier contenant les .tif
print("Dossier des rasters :", DOSSIER)

In [ ]:
import os, re, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
import rasterio
from rasterio.merge import merge
from rasterio.transform import xy

BANDES = globals().get("BANDES", {})
SORTIE = os.path.join(DOSSIER, "tables")
os.makedirs(SORTIE, exist_ok=True)
LIBELLES = {0: "non inondé", 1: "inondé", 255: "sans donnée"}

def est_masque(nom):
    return nom.startswith("masque_")

def noms_rasters():
    base = {re.sub(r"-\d{10}-\d{10}$", "", os.path.basename(f)[:-4]) for f in glob.glob(os.path.join(DOSSIER, "*.tif"))}
    return sorted(base)

def lire_raster(nom):
    fichiers = sorted(glob.glob(os.path.join(DOSSIER, nom + ".tif")) + glob.glob(os.path.join(DOSSIER, nom + "-*.tif")))
    srcs = [rasterio.open(f) for f in fichiers]
    try:
        if len(srcs) == 1:
            arr, tr = srcs[0].read(), srcs[0].transform
        else:
            arr, tr = merge(srcs)     # GEE découpe parfois l'export en plusieurs tuiles
        r = dict(arr=arr, transform=tr, crs=srcs[0].crs, res=srcs[0].res, dtype=srcs[0].dtypes[0],
                 descr=list(srcs[0].descriptions), fichiers=[os.path.basename(f) for f in fichiers])
    finally:
        for s in srcs:
            s.close()
    return r

def noms_bandes(nom, r):
    if nom in BANDES:
        return BANDES[nom]
    return [d or f"bande_{i + 1}" for i, d in enumerate(r["descr"])]

def table_bandes(nom, r):
    nd = 255 if est_masque(nom) else -9999
    lignes = []
    for i, b in enumerate(noms_bandes(nom, r)):
        a = r["arr"][i].astype("float64")
        a[(a == nd) | ~np.isfinite(a)] = np.nan
        v = a[np.isfinite(a)]
        lignes.append({"raster": nom, "bande": b, "pixels_total": a.size, "pixels_valides": v.size,
                       "% valides": round(v.size / a.size * 100, 2),
                       "min": float(v.min()) if v.size else np.nan, "max": float(v.max()) if v.size else np.nan,
                       "moyenne": float(v.mean()) if v.size else np.nan,
                       "mediane": float(np.median(v)) if v.size else np.nan,
                       "ecart_type": float(v.std()) if v.size else np.nan})
    return pd.DataFrame(lignes)

def table_valeurs(nom, r):
    vals, cnt = np.unique(r["arr"][0], return_counts=True)
    aire = abs(r["res"][0] * r["res"][1])
    df = pd.DataFrame({"raster": nom, "Value": vals, "Count": cnt})
    df["Libelle"] = df["Value"].map(LIBELLES).fillna("autre")
    df["Surface_km2"] = (df["Count"] * aire / 1e6).round(3)
    valides = df.loc[df["Value"] != 255, "Count"].sum()
    df["% des pixels valides"] = np.where(df["Value"] == 255, np.nan, (df["Count"] / valides * 100).round(2))
    return df

def echantillon_pixels(nom, r, n=5000, seed=42):
    nd = 255 if est_masque(nom) else -9999
    arr = r["arr"]
    _, H, W = arr.shape
    rng = np.random.default_rng(seed)
    flat = rng.choice(H * W, min(n, H * W), replace=False)
    lig, col = np.divmod(flat, W)
    xs, ys = xy(r["transform"], lig, col)
    df = pd.DataFrame({"x": np.asarray(xs), "y": np.asarray(ys)})
    for i, b in enumerate(noms_bandes(nom, r)):
        v = arr[i, lig, col].astype("float64")
        v[v == nd] = np.nan
        df[b] = v
    return df

### Génération des tables pour chaque raster

In [ ]:
liste = noms_rasters()
print(len(liste), "rasters trouvés :", liste)

T_BANDES, T_VALEURS, ECH, INFOS = [], [], {}, []
for nom in liste:
    r = lire_raster(nom)
    t = table_bandes(nom, r)
    T_BANDES.append(t)
    if est_masque(nom):
        T_VALEURS.append(table_valeurs(nom, r))
    ECH[nom] = echantillon_pixels(nom, r)
    ECH[nom].to_csv(os.path.join(SORTIE, f"{nom}_pixels.csv"), index=False)
    INFOS.append({"raster": nom, "bandes": r["arr"].shape[0], "lignes": r["arr"].shape[1], "colonnes": r["arr"].shape[2],
                  "CRS": str(r["crs"]), "pixel_m": round(abs(r["res"][0]), 2), "type": r["dtype"],
                  "tuiles": len(r["fichiers"])})
    del r

df_infos = pd.DataFrame(INFOS)
df_bandes = pd.concat(T_BANDES, ignore_index=True)
df_valeurs = pd.concat(T_VALEURS, ignore_index=True) if T_VALEURS else pd.DataFrame()

print("\n== Informations générales =="); display(df_infos)
print("\n== Table des bandes (statistiques par raster) =="); display(df_bandes.round(4))
if len(df_valeurs):
    print("\n== Table des valeurs des masques (type table attributaire ArcGIS) =="); display(df_valeurs)

In [ ]:
# Comparaison avant / après (moyenne de chaque bande)
st = df_bandes.copy()
st["phase"] = st["raster"].str.extract(r"_(avant|apres)$")[0]
st["groupe"] = st["raster"].str.replace(r"_(avant|apres)$", "", regex=True)
st = st.dropna(subset=["phase"])
if len(st):
    cmp = st.pivot_table(index=["groupe", "bande"], columns="phase", values="moyenne")
    if {"avant", "apres"} <= set(cmp.columns):
        cmp["variation (après - avant)"] = cmp["apres"] - cmp["avant"]
    display(cmp.round(4))

In [ ]:
# Export Excel (une feuille par type de table + un échantillon de pixels par raster) et CSV
chemin_xlsx = os.path.join(SORTIE, "tables_attributaires.xlsx")
with pd.ExcelWriter(chemin_xlsx, engine="openpyxl") as xw:
    df_infos.to_excel(xw, sheet_name="infos", index=False)
    df_bandes.to_excel(xw, sheet_name="bandes", index=False)
    if len(df_valeurs):
        df_valeurs.to_excel(xw, sheet_name="valeurs_masques", index=False)
    index_feuilles = []
    for i, (nom, df) in enumerate(ECH.items()):
        feuille = f"px_{i:02d}"
        df.to_excel(xw, sheet_name=feuille, index=False)
        index_feuilles.append({"feuille": feuille, "raster": nom})
    pd.DataFrame(index_feuilles).to_excel(xw, sheet_name="index_pixels", index=False)
df_bandes.to_csv(os.path.join(SORTIE, "bandes.csv"), index=False)
if len(df_valeurs):
    df_valeurs.to_csv(os.path.join(SORTIE, "valeurs_masques.csv"), index=False)
print("Tables enregistrées dans :", SORTIE)

### Jeu de données prêt pour les modèles
Les deux autres notebooks attendent des fichiers séparés (`dem.tif`, `slope.tif`, `ndvi.tif`, `ndbi.tif`, `dist_eau.tif` et un masque `flood_XXXX.tif`). Cette cellule les crée pour chaque événement, dans `dataset_2021` et `dataset_2010`.
- `ndvi` et `ndbi` viennent de l'image **avant** l'événement (jamais de l'image après, pour éviter une fuite d'information).
- `dist_eau` = distance (en mètres) à l'eau permanente (JRC), calculée ici.
- Ordre des bandes de mes exports : composites optiques = bleu, vert, rouge, nir, swir1, swir2, ndvi, ndbi, mndwi ; `dem_pente` = dem, pente.

In [ ]:
from scipy.ndimage import distance_transform_edt

def ecrire_tif(chemin, arr, r, nodata=-9999.0):
    a = arr.astype("float32")
    p = dict(driver="GTiff", height=a.shape[0], width=a.shape[1], count=1, dtype="float32",
             crs=r["crs"], transform=r["transform"], nodata=nodata, compress="lzw")
    with rasterio.open(chemin, "w", **p) as dst:
        dst.write(np.where(np.isfinite(a), a, nodata), 1)

def bande(nom, i):
    r = lire_raster(nom)
    a = r["arr"][i].astype("float32")
    a[a == -9999] = np.nan
    return a, r

def construire_dataset(dossier_sortie, optique, masque, nom_masque):
    os.makedirs(dossier_sortie, exist_ok=True)
    a, r = bande("dem_pente", 0); ecrire_tif(f"{dossier_sortie}/dem.tif", a, r)
    a, r = bande("dem_pente", 1); ecrire_tif(f"{dossier_sortie}/slope.tif", a, r)
    a, r = bande(optique, 6); ecrire_tif(f"{dossier_sortie}/ndvi.tif", a, r)
    a, r = bande(optique, 7); ecrire_tif(f"{dossier_sortie}/ndbi.tif", a, r)
    eau, r = bande("eau_permanente", 0)
    eau = np.nan_to_num(eau) == 1
    if eau.sum() == 0:
        print("⚠ Aucun pixel d'eau permanente trouvé : dist_eau non créée.")
    else:
        ecrire_tif(f"{dossier_sortie}/dist_eau.tif", distance_transform_edt(~eau) * abs(r["res"][0]), r)
    m = lire_raster(masque)
    mk = m["arr"][0].astype("float32"); mk[mk == 255] = np.nan
    ecrire_tif(f"{dossier_sortie}/{nom_masque}.tif", mk, m)
    print("Jeu de données créé :", dossier_sortie, sorted(os.listdir(dossier_sortie)))

dispo = set(noms_rasters())
if {"dem_pente", "eau_permanente", "2021_janvier_s2_avant", "masque_2021_janvier"} <= dispo:
    construire_dataset(os.path.join(DOSSIER, "dataset_2021"), "2021_janvier_s2_avant", "masque_2021_janvier", "flood_2021")
if {"dem_pente", "eau_permanente", "2010_novembre_l5_avant", "masque_2010_novembre"} <= dispo:
    construire_dataset(os.path.join(DOSSIER, "dataset_2010"), "2010_novembre_l5_avant", "masque_2010_novembre", "flood_2010")

### Aperçu visuel (première bande de chaque raster)

In [ ]:
n = len(liste)
cols_ = 4
lignes_ = int(np.ceil(n / cols_))
fig, axes = plt.subplots(lignes_, cols_, figsize=(4 * cols_, 3.2 * lignes_))
axes = np.atleast_1d(axes).ravel()
for ax, nom in zip(axes, liste):
    r = lire_raster(nom)
    nd = 255 if est_masque(nom) else -9999
    a = r["arr"][0][::4, ::4].astype("float64")
    a[a == nd] = np.nan
    ax.imshow(a, cmap="Blues" if est_masque(nom) else "viridis")
    ax.set_title(nom, fontsize=8); ax.axis("off")
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## Pour la suite
- Dans les notebooks de modélisation, mettez `DEMO = False`, `DATA_DIR` = dossier `dataset_2021` (ou `dataset_2010`) et `FLOOD_MASK = flood_2021.tif` (ou `flood_2010.tif`).
- Les rasters `*_avant`, `*_apres`, `dem_pente` et le masque correspondent directement au notebook de prédiction (variables explicatives + cible).
- Si le masque est utilisé comme cible, ne mettez pas en variable explicative l'indice qui a servi à le créer (MNDWI pour Landsat, VV pour Sentinel-1 *après*), sinon le modèle retrouve la règle de construction. Utilisez les variables *avant* l'événement, le MNT, la pente, NDVI et NDBI.
- Les seuils des masques sont des valeurs de départ : validez-les visuellement et remplacez-les par votre propre méthode si elle est meilleure.